In [2]:
!wget -q https://files.grouplens.org/datasets/movielens/ml-latest-small.zip
!unzip -q -o ml-latest-small.zip
import pandas as pd

ratings = pd.read_csv("ml-latest-small/ratings.csv")
movies = pd.read_csv("ml-latest-small/movies.csv")

print(ratings.shape, movies.shape)
print(ratings.head())
print(movies.head())
print(ratings["userId"].nunique(), ratings["movieId"].nunique())
print(ratings["rating"].describe())

(100836, 4) (9742, 3)
   userId  movieId  rating  timestamp
0       1        1     4.0  964982703
1       1        3     4.0  964981247
2       1        6     4.0  964982224
3       1       47     5.0  964983815
4       1       50     5.0  964982931
   movieId                               title  \
0        1                    Toy Story (1995)   
1        2                      Jumanji (1995)   
2        3             Grumpier Old Men (1995)   
3        4            Waiting to Exhale (1995)   
4        5  Father of the Bride Part II (1995)   

                                        genres  
0  Adventure|Animation|Children|Comedy|Fantasy  
1                   Adventure|Children|Fantasy  
2                               Comedy|Romance  
3                         Comedy|Drama|Romance  
4                                       Comedy  
610 9724
count    100836.000000
mean          3.501557
std           1.042529
min           0.500000
25%           3.000000
50%           3.500000
75%     

In [3]:
import numpy as np

ratings = ratings.sort_values(["userId", "timestamp"]).reset_index(drop=True)
ratings["rank"] = ratings.groupby("userId").cumcount()
ratings["n_user"] = ratings.groupby("userId")["rating"].transform("size")

is_test = ratings["rank"] >= 0.8 * ratings["n_user"]
train = ratings[~is_test].copy()
test = ratings[is_test].copy()

print("Train:", len(train), "| Test:", len(test))
print("Users in train:", train["userId"].nunique(), "| in test:", test["userId"].nunique())

unseen = ~test["movieId"].isin(set(train["movieId"]))
print("Test ratings of movies never rated in train:", unseen.sum(), "(", round(unseen.mean() * 100, 1), "% )")

density = len(ratings) / (ratings["userId"].nunique() * ratings["movieId"].nunique())
print("Filled cells in the user-movie table:", round(density * 100, 2), "%")

per_user = ratings.groupby("userId").size()
per_movie = ratings.groupby("movieId").size()
print(per_user.describe())
print(per_movie.describe())
print((per_movie == 1).sum(), "movies have only 1 rating")

print("Average rating, train:", round(train["rating"].mean(), 3), "| test:", round(test["rating"].mean(), 3))

Train: 80896 | Test: 19940
Users in train: 610 | in test: 610
Test ratings of movies never rated in train: 1682 ( 8.4 % )
Filled cells in the user-movie table: 1.7 %
count     610.000000
mean      165.304918
std       269.480584
min        20.000000
25%        35.000000
50%        70.500000
75%       168.000000
max      2698.000000
dtype: float64
count    9724.000000
mean       10.369807
std        22.401005
min         1.000000
25%         1.000000
50%         3.000000
75%         9.000000
max       329.000000
dtype: float64
3446 movies have only 1 rating
Average rating, train: 3.514 | test: 3.451


In [4]:
mu = train["rating"].mean()

user_mean = train.groupby("userId")["rating"].mean()
movie_mean = train.groupby("movieId")["rating"].mean()

movie_bias = (train["rating"] - mu).groupby(train["movieId"]).sum() / (train.groupby("movieId").size() + 5)

train["resid"] = train["rating"] - mu - train["movieId"].map(movie_bias)
user_bias = train.groupby("userId")["resid"].sum() / (train.groupby("userId").size() + 10)

y = test["rating"].values
preds = {
    "Global average": np.full(len(test), mu),
    "Movie average": test["movieId"].map(movie_mean).fillna(mu).values,
    "User average": test["userId"].map(user_mean).fillna(mu).values,
    "Baseline model (user + movie bias)": (mu + test["movieId"].map(movie_bias).fillna(0)
                                           + test["userId"].map(user_bias).fillna(0)).values,
}

rmse = lambda p: np.sqrt(np.mean((y - np.clip(p, 0.5, 5)) ** 2))
mae = lambda p: np.mean(np.abs(y - np.clip(p, 0.5, 5)))

results = pd.DataFrame({name: [rmse(p), mae(p)] for name, p in preds.items()},
                       index=["RMSE", "MAE"]).T.round(4)
print("Global average rating:", round(mu, 3))
print(results)

Global average rating: 3.514
                                      RMSE     MAE
Global average                      1.0688  0.8360
Movie average                       1.0227  0.7867
User average                        0.9648  0.7486
Baseline model (user + movie bias)  0.8990  0.6926


In [5]:
from scipy.sparse import csr_matrix
from scipy.sparse.linalg import svds

train["base"] = mu + train["movieId"].map(movie_bias) + train["userId"].map(user_bias)
train["res"] = train["rating"] - train["base"]
test["base"] = mu + test["movieId"].map(movie_bias).fillna(0) + test["userId"].map(user_bias).fillna(0)

users = np.sort(train["userId"].unique())
movies_tr = np.sort(train["movieId"].unique())
u_idx = {u: i for i, u in enumerate(users)}
m_idx = {m: i for i, m in enumerate(movies_tr)}

R = csr_matrix((train["res"].values,
                (train["userId"].map(u_idx).values, train["movieId"].map(m_idx).values)),
               shape=(len(users), len(movies_tr)))

t_rows = test["userId"].map(u_idx).values
t_cols_raw = test["movieId"].map(m_idx)
known = t_cols_raw.notna().values
t_cols = t_cols_raw.fillna(0).astype(int).values
base_test = test["base"].values

svd_preds = {}
for k in [5, 10, 20, 40]:
    U, s, Vt = svds(R, k=k)
    extra = np.sum(U[t_rows] * s * Vt[:, t_cols].T, axis=1)
    svd_preds[k] = base_test + np.where(known, extra, 0)
    print("SVD with", k, "factors | RMSE:", round(rmse(svd_preds[k]), 4), "| MAE:", round(mae(svd_preds[k]), 4))

print("Baseline model          | RMSE:", round(rmse(base_test), 4), "| MAE:", round(mae(base_test), 4))

SVD with 5 factors | RMSE: 0.8949 | MAE: 0.6889
SVD with 10 factors | RMSE: 0.8951 | MAE: 0.6887
SVD with 20 factors | RMSE: 0.8939 | MAE: 0.6881
SVD with 40 factors | RMSE: 0.8948 | MAE: 0.6886
Baseline model          | RMSE: 0.899 | MAE: 0.6926


In [6]:
from sklearn.metrics.pairwise import cosine_similarity

sim = cosine_similarity(R.T.tocsr()).astype(np.float32)
np.fill_diagonal(sim, 0)

K = 30
item_extra = np.zeros(len(test))
for n in range(len(test)):
    if not known[n]:
        continue
    u, m = t_rows[n], t_cols[n]
    start, end = R.indptr[u], R.indptr[u + 1]
    rated = R.indices[start:end]
    resid = R.data[start:end]
    s = sim[m, rated]
    if len(s) > K:
        top = np.argpartition(-s, K)[:K]
        s, resid = s[top], resid[top]
    keep = s > 0
    if keep.any():
        item_extra[n] = np.sum(s[keep] * resid[keep]) / (s[keep].sum() + 1)

item_pred = base_test + item_extra
print("Item-based | RMSE:", round(rmse(item_pred), 4), "| MAE:", round(mae(item_pred), 4))

best_k = min(svd_preds, key=lambda k: rmse(svd_preds[k]))
final = {"Baseline model": base_test,
         "SVD, " + str(best_k) + " factors": svd_preds[best_k],
         "Item-based neighbors": item_pred,
         "Average of SVD and item-based": (svd_preds[best_k] + item_pred) / 2}
table = pd.DataFrame({name: [rmse(p), mae(p)] for name, p in final.items()},
                     index=["RMSE", "MAE"]).T.round(4)
print(table)

Item-based | RMSE: 0.8807 | MAE: 0.673
                                 RMSE     MAE
Baseline model                 0.8990  0.6926
SVD, 20 factors                0.8939  0.6881
Item-based neighbors           0.8807  0.6730
Average of SVD and item-based  0.8844  0.6781


In [7]:
titles = movies.set_index("movieId")["title"]
counts = train.groupby("movieId").size()
cnt_vec = counts.reindex(movies_tr).values
bias_vec = movie_bias.reindex(movies_tr).values
K = 30

def recommend_ids(user_id, n=10, min_ratings=20):
    u = u_idx[user_id]
    start, end = R.indptr[u], R.indptr[u + 1]
    rated = R.indices[start:end]
    resid = R.data[start:end]
    S = sim[:, rated]
    if S.shape[1] > K:
        top = np.argpartition(-S, K, axis=1)[:, :K]
        S_top = np.take_along_axis(S, top, axis=1)
        r_top = resid[top]
    else:
        S_top = S
        r_top = np.tile(resid, (S.shape[0], 1))
    S_top = np.where(S_top > 0, S_top, 0)
    extra = (S_top * r_top).sum(axis=1) / (S_top.sum(axis=1) + 1)
    score = mu + bias_vec + user_bias[user_id] + extra
    score[rated] = -np.inf
    score[cnt_vec < min_ratings] = -np.inf
    best = np.argsort(-score)[:n]
    return movies_tr[best], score[best]

def show_recommendations(user_id, n=10):
    liked = train[train["userId"] == user_id].sort_values("rating", ascending=False).head(8)
    print("User", user_id, "- movies rated highly in training:")
    for m, r in zip(liked["movieId"], liked["rating"]):
        print("  ", r, titles[m])
    ids, scores = recommend_ids(user_id, n)
    print("Recommended movies:")
    for m, s in zip(ids, scores):
        print("  ", round(min(s, 5), 2), titles[m])

show_recommendations(1)

User 1 - movies rated highly in training:
   5.0 Star Wars: Episode VI - Return of the Jedi (1983)
   5.0 Bambi (1942)
   5.0 Bottle Rocket (1996)
   5.0 Gladiator (2000)
   5.0 South Park: Bigger, Longer and Uncut (1999)
   5.0 Wayne's World (1992)
   5.0 SLC Punk! (1998)
   5.0 Big Lebowski, The (1998)
Recommended movies:
   5 Sunset Blvd. (a.k.a. Sunset Boulevard) (1950)
   5 Good, the Bad and the Ugly, The (Buono, il brutto, il cattivo, Il) (1966)
   5 Bridge on the River Kwai, The (1957)
   5 Cool Hand Luke (1967)
   5 Shawshank Redemption, The (1994)
   5 Lawrence of Arabia (1962)
   5 North by Northwest (1959)
   5 Casablanca (1942)
   5 Philadelphia Story, The (1940)
   5 Godfather: Part II, The (1974)


In [8]:
rel = test[test["rating"] >= 4].groupby("userId")["movieId"].apply(set)
train_sets = train.groupby("userId")["movieId"].apply(set)
pop_order = counts[counts >= 20].sort_values(ascending=False).index.values

hits_cf, hits_pop = [], []
for uid in rel.index:
    ids_cf, _ = recommend_ids(uid, 10)
    ids_pop = [m for m in pop_order if m not in train_sets[uid]][:10]
    hits_cf.append(len(set(ids_cf) & rel[uid]) / 10)
    hits_pop.append(len(set(ids_pop) & rel[uid]) / 10)

print("Users tested:", len(rel))
print("Precision@10, most popular movies:", round(np.mean(hits_pop), 4))
print("Precision@10, item-based model   :", round(np.mean(hits_cf), 4))

Users tested: 591
Precision@10, most popular movies: 0.056
Precision@10, item-based model   : 0.044
